# EightChess NNUE fine-tune (2x T4)

Streams all mounted .npz shards (t91 + Stockfish binpacks + Lc0), fine-tunes the
champion network, checkpoints every N shards, and exports engine-ready `.nnue`.


In [ ]:
import glob, os, subprocess, sys, time, shutil, zipfile
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout, flush=True)

try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-chess"])
except Exception as e:
    print("pip install error:", e, flush=True)

dst = "/kaggle/working/eightchess_nnue"

# 1) Prefer a mounted source directory
code_dir = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.py" in files and "model.py" in files and "__init__.py" in files:
        code_dir = root
        break
print("code_dir:", code_dir, flush=True)

if code_dir:
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(code_dir, dst)
    print("copied code dir to", dst, flush=True)
else:
    # 2) Fall back to the packaged zip
    zips = [z for z in glob.glob("/kaggle/input/**/*.zip", recursive=True) if "eightchess_nnue" in os.path.basename(z)]
    print("code zips:", zips, flush=True)
    if zips:
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/kaggle/working")
        print("extracted", zips[0], "-> /kaggle/working", flush=True)

print("package exists:", os.path.isdir(dst), flush=True)
if os.path.isdir(dst):
    print("package files:", sorted(os.listdir(dst)), flush=True)
sys.path.insert(0, "/kaggle/working")
import eightchess_nnue
print("training code:", eightchess_nnue.__file__, flush=True)

try:
    import chess
    print("python-chess", chess.__version__, flush=True)
except ImportError as e:
    print("chess import failed:", e, flush=True)


In [ ]:
import glob, os, numpy as np
def find(p):
    return sorted(glob.glob(p, recursive=True))

# Find all directories containing .npz shards
npz_dirs = set()
for root, dirs, files in os.walk("/kaggle/input"):
    if any(f.endswith(".npz") for f in files):
        npz_dirs.add(root)
npz_dirs = sorted(npz_dirs)
print("npz dirs:", npz_dirs, flush=True)

sources = {}
for d in npz_dirs:
    name = os.path.basename(d).lower()
    n = len(glob.glob(os.path.join(d, "*.npz")))
    sources[name] = n
    print(f"  {d}: {n} shards", flush=True)

all_shards = []
for d in npz_dirs:
    all_shards += sorted(glob.glob(os.path.join(d, "*.npz")))
print("total shards:", len(all_shards), flush=True)
sizes = []
for p in all_shards[:: max(1, len(all_shards) // 40)]:
    with np.load(p) as f:
        sizes.append(int(f["bucket"].shape[0]))
mean = int(np.mean(sizes)) if sizes else 0
print(f"approx positions: {mean * len(all_shards):,}", flush=True)
# Store for cell 3
import json as _j
with open("/kaggle/working/_dirs.json", "w") as f:
    _j.dump(npz_dirs, f)


In [ ]:
import subprocess, sys, os, json
BASE = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in sorted(files):
        if f.endswith(".nnue"):
            BASE = os.path.join(root, f)
            break
    if BASE:
        break
print("BASE:", BASE, flush=True)

with open("/kaggle/working/_dirs.json") as f:
    dirs = json.load(f)
print("data dirs:", dirs, flush=True)

cmd = [sys.executable, "-m", "eightchess_nnue.train",
       "--data", *dirs,
       "--base", BASE,
       "--out-dir", "/kaggle/working/ckpt",
       "--epochs", "2",
       "--batch-size", "4096",
       "--lr", "1e-4",
       "--ft-lr-scale", "1.0",
       "--head-lr-scale", "0.1",
       "--revive-new-channels",
       "--freeze-backbone",
       "--val-frac", "0.02",
       "--save-every-shards", "50",
       "--max-seconds", "25200",
       "--export-every", "1"]
print(" ".join(cmd), flush=True)
env = dict(os.environ)
env["PYTHONPATH"] = "/kaggle/working:" + env.get("PYTHONPATH", "")
subprocess.check_call(cmd, cwd="/kaggle/working", env=env)


In [ ]:
import glob, os, shutil
nns = sorted(glob.glob("/kaggle/working/ckpt/*.nnue"), key=os.path.getmtime)
print("exported networks:", nns, flush=True)
if nns:
    shutil.copy(nns[-1], "/kaggle/working/eightchess_finetuned.nnue")
    print("final:", "/kaggle/working/eightchess_finetuned.nnue", flush=True)
